# QuantumSynapse Fabric — Qiskit verification notebook (Google Colab)This notebook re-derives, with **real Qiskit**, the same physics the project'sDeno statevector engine (`supabase/functions/_shared/statevector.ts`) computes:1. Quantum RNG (Hadamard-basis measurement)2. GHZ entanglement — only all-zeros / all-ones outcomes3. BB84 QKD — QBER ~0% clean, ~25% with an intercept-resend eavesdropper4. CHSH / Bell score — must land in `2.0 < S <= 2.828` (the LeviathanCoin governance window)5. VQE on the transverse-field Ising model, checked against exact diagonalisationNothing here touches a blockchain, a private key, or a wallet. It is a physicscross-check only.**Run order:** top to bottom. Cell 1 installs Qiskit in Colab.

In [ ]:
# Colab install. Skip if Qiskit is already present locally.%pip install -q "qiskit>=1.2" "qiskit-aer>=0.15" numpy

In [ ]:
import numpy as npfrom qiskit import QuantumCircuit, transpilefrom qiskit_aer import AerSimulatorfrom qiskit.quantum_info import Statevectorsim = AerSimulator()rng = np.random.default_rng(20260824)print("Qiskit ready. Aer backend:", sim.name)

## 1. Quantum RNG — Hadamard-basis measurement

In [ ]:
def qrng_bits(n_bits: int) -> str:    qc = QuantumCircuit(1, 1)    qc.h(0)    qc.measure(0, 0)    counts = sim.run(transpile(qc, sim), shots=n_bits, memory=True).result().get_memory()    return "".join(counts)bits = qrng_bits(4096)ones = bits.count("1") / len(bits)print("first 64 bits:", bits[:64])print(f"fraction of ones: {ones:.4f}  (expected ~0.5)")assert 0.45 < ones < 0.55, "QRNG is not balanced — check the H gate"print("PASS: balanced within tolerance")

## 2. GHZ state — perfect multi-qubit correlation

In [ ]:
def ghz(n: int) -> QuantumCircuit:    qc = QuantumCircuit(n, n)    qc.h(0)    for k in range(1, n):        qc.cx(0, k)    qc.measure(range(n), range(n))    return qcfor n in (2, 3, 4):    counts = sim.run(transpile(ghz(n), sim), shots=2048).result().get_counts()    allowed = {"0" * n, "1" * n}    assert set(counts) <= allowed, f"GHZ-{n} leaked an uncorrelated outcome: {counts}"    print(f"GHZ-{n}: {counts}  -> only {sorted(allowed)} occur  PASS")

## 3. BB84 QKD — QBER with and without an eavesdropperAlice encodes random bits in random bases; Bob measures in random bases; theykeep only the positions where the bases matched. Eve does intercept-resend:she measures in a random basis and re-prepares, which corrupts ~25% of thesifted key. That 25% is the physical signature the project's BB84 endpointreports.

In [ ]:
def bb84(n_bits: int = 4096, eavesdropper: bool = False):    a_bits  = rng.integers(0, 2, n_bits)    a_bases = rng.integers(0, 2, n_bits)   # 0 = Z, 1 = X    b_bases = rng.integers(0, 2, n_bits)    e_bases = rng.integers(0, 2, n_bits)    bob_bits = np.empty(n_bits, dtype=int)    for i in range(n_bits):        qc = QuantumCircuit(1, 1)        if a_bits[i]:            qc.x(0)        if a_bases[i]:            qc.h(0)                     # prepare in the X basis        if eavesdropper:            if e_bases[i]:                qc.h(0)            qc.measure(0, 0)            # Eve's projective measurement            qc.reset(0)                 # ...then she re-prepares what she saw            qc.x(0).c_if(qc.cregs[0], 1)            if e_bases[i]:                qc.h(0)        if b_bases[i]:            qc.h(0)                     # Bob measures in the X basis        qc.measure(0, 0)        bob_bits[i] = int(sim.run(transpile(qc, sim), shots=1, memory=True)                          .result().get_memory()[0])    sifted = a_bases == b_bases    errors = (a_bits[sifted] != bob_bits[sifted]).sum()    key_len = int(sifted.sum())    return {"sifted_key_length": key_len, "qber": errors / key_len if key_len else 0.0}clean = bb84(768, eavesdropper=False)print("no eavesdropper:", clean, f"QBER = {clean['qber']*100:.2f}%")assert clean["qber"] < 0.02spied = bb84(768, eavesdropper=True)print("intercept-resend:", spied, f"QBER = {spied['qber']*100:.2f}%")assert 0.15 < spied["qber"] < 0.35, "eavesdropper should push QBER toward 25%"print("PASS: QBER ~0% clean, ~25% under attack — matches the Deno engine")

## 4. CHSH / Bell score — the LeviathanCoin governance window`src/contracts/LeviathanCoin.sol` accepts an attestation only when`2.0 < S <= 2.828`: above the classical limit, at or below Tsirelson's bound`2*sqrt(2)`. This cell measures S on a real Bell pair.

In [ ]:
def chsh_term(theta_a, theta_b, shots=8192):    qc = QuantumCircuit(2, 2)    qc.h(0); qc.cx(0, 1)    qc.ry(-2 * theta_a, 0)    qc.ry(-2 * theta_b, 1)    qc.measure([0, 1], [0, 1])    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()    e = sum((1 if k.count("1") % 2 == 0 else -1) * v for k, v in counts.items())    return e / shotsa0, a1 = 0.0, np.pi / 4b0, b1 = np.pi / 8, 3 * np.pi / 8S = abs(chsh_term(a0, b0) - chsh_term(a0, b1) + chsh_term(a1, b0) + chsh_term(a1, b1))tsirelson = 2 * np.sqrt(2)print(f"measured S      = {S:.4f}")print(f"classical limit = 2.0000")print(f"Tsirelson bound = {tsirelson:.4f}")accepted = 2.0 < S <= tsirelson + 1e-9print("on-chain verdict:", "ACCEPTED" if accepted else "REJECTED")assert accepted, "S outside the governance window — attestation would be rejected on-chain"print("PASS: S is inside 2.0 < S <= 2.828")

## 5. VQE — transverse-field Ising model vs exact diagonalisationMirrors `_shared/variational.ts`. The Qiskit result should match the exactground-state energy to a few decimal places.

In [ ]:
from scipy.optimize import minimizefrom qiskit.quantum_info import SparsePauliOpn, J, h, layers = 3, 1.0, 1.0, 2terms = [("Z" * 0 + "".join("Z" if q in (i, i + 1) else "I" for q in range(n)), -J)         for i in range(n - 1)]terms += [("".join("X" if q == i else "I" for q in range(n)), -h) for i in range(n)]H = SparsePauliOp.from_list(terms)exact = float(np.min(np.linalg.eigvalsh(H.to_matrix())))def ansatz(params):    qc = QuantumCircuit(n)    k = 0    for _ in range(layers):        for q in range(n):            qc.ry(params[k], q); k += 1        for q in range(n - 1):            qc.cx(q, q + 1)    for q in range(n):        qc.ry(params[k], q); k += 1    return qcn_params = n * (layers + 1)energy = lambda p: float(np.real(Statevector(ansatz(p)).expectation_value(H)))res = minimize(energy, rng.uniform(0, 2 * np.pi, n_params), method="COBYLA",               options={"maxiter": 800})print(f"VQE energy   = {res.fun:.6f}")print(f"exact energy = {exact:.6f}")print(f"error        = {abs(res.fun - exact):.2e}")assert abs(res.fun - exact) < 1e-2print("PASS: VQE converged to the exact ground state")

## Summary| Check | Expectation | Source of truth in this repo || --- | --- | --- || QRNG balance | ~0.5 ones | `/v1/quantum/qrng` || GHZ correlation | only `000…` / `111…` | `/v1/quantum/entangle` || BB84 QBER | ~0% clean, ~25% spied | `_shared/bb84.ts` || CHSH S | `2.0 < S <= 2.828` | `LeviathanCoin.submitAttestation` || VQE energy | matches exact diagonalisation | `_shared/variational.ts` |Every assertion above must pass. A failure means the physics assumption behindthe corresponding service changed and the service needs re-checking — not thatthe notebook should be loosened.